In [3]:
import sys
import logging
from typing import Dict, Any, List, Optional
from dataclasses import dataclass
from enum import Enum

# 配置日志输出
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("AgentArchitecture")

# ============================================================================
# 1. 契约层：错误与业务 Outcome 分离模型 (Contract Layer)
# ============================================================================

class ToolErrorKind(str, Enum):
    TIMEOUT = "TIMEOUT"
    RATE_LIMITED = "RATE_LIMITED"
    SERVICE_UNAVAILABLE = "SERVICE_UNAVAILABLE"
    INPUT_VALIDATION_ERROR = "INPUT_VALIDATION_ERROR"
    OUTPUT_VALIDATION_ERROR = "OUTPUT_VALIDATION_ERROR"

@dataclass
class ToolError:
    kind: ToolErrorKind
    message: str
    retryable: bool = False

class BusinessStatus(str, Enum):
    FOUND = "FOUND"
    EMPTY = "EMPTY"
    AMBIGUOUS = "AMBIGUOUS"
    APPROVED = "APPROVED"
    DECLINED = "DECLINED"

@dataclass
class BusinessOutcome:
    status: BusinessStatus
    data: Any = None
    reason: Optional[str] = None

@dataclass
class ToolResult:
    ok: bool  # True 代表工具执行、网络、Schema校验全通过；False 代表执行中断/故障
    data: Optional[BusinessOutcome] = None
    error: Optional[ToolError] = None

# ============================================================================
# 2. Mock 业务工具集 (Mock Tools Engine)
# ============================================================================

MOCK_DB = {
    "employees": [
        {"emp_id": "EMP_1001", "name": "Kevin Zhang", "department": "IT", "leave_balance": 12},
        {"emp_id": "EMP_2038", "name": "Kevin Wang", "department": "Sales", "leave_balance": 8},
    ]
}

def search_employee(name: str) -> ToolResult:
    """搜索员工工具：业务层正常处理 EMPTY / AMBIGUOUS / FOUND"""
    logger.info(f"⚙️ [Tool Execution] search_employee(name='{name}')")
    matches = [emp for emp in MOCK_DB["employees"] if name.lower() in emp["name"].lower()]
    
    if len(matches) == 0:
        return ToolResult(ok=True, data=BusinessOutcome(status=BusinessStatus.EMPTY, data=[]))
    elif len(matches) > 1:
        # 已修正：使用 BusinessStatus.AMBIGUOUS
        return ToolResult(
            ok=True, 
            data=BusinessOutcome(status=BusinessStatus.AMBIGUOUS, data=matches, reason="找到多个同名员工")
        )
    return ToolResult(ok=True, data=BusinessOutcome(status=BusinessStatus.FOUND, data=matches[0]))

def process_payment(account_id: str, amount: float, simulate_timeout: bool = False) -> ToolResult:
    """扣款工具：区分 Timeout (ok=False) 与 余额不足 (ok=True, status=DECLINED)"""
    logger.info(f"⚙️ [Tool Execution] process_payment(account_id='{account_id}', amount={amount})")
    
    # 模拟真正的 Runtime 故障
    if simulate_timeout:
        logger.error("💥 [Runtime Fault] Payment Gateway Socket Timeout Exception!")
        return ToolResult(
            ok=False, 
            error=ToolError(kind=ToolErrorKind.TIMEOUT, message="Gateway Read Timeout", retryable=True)
        )
    
    # 模拟业务层拒绝 (余额不足)
    if amount > 1000:
        logger.warning("⚠️ [Business Outcome] Payment Declined due to Insufficient Funds")
        return ToolResult(
            ok=True, 
            data=BusinessOutcome(
                status=BusinessStatus.DECLINED, 
                data={"account_id": account_id, "amount": amount}, 
                reason="INSUFFICIENT_FUNDS"
            )
        )
        
    return ToolResult(
        ok=True, 
        data=BusinessOutcome(status=BusinessStatus.APPROVED, data={"transaction_id": "TXN_99812", "amount": amount})
    )

# ============================================================================
# 3. Guard Policy：Goal Mutation Guard (防止意图静默篡改)
# ============================================================================

class PolicyDecision(str, Enum):
    ALLOW = "ALLOW"
    REQUIRE_USER_CONFIRMATION = "REQUIRE_USER_CONFIRMATION"

@dataclass
class GoalMutationGuard:
    @staticmethod
    def evaluate(original_goal: Dict[str, Any], proposed_action: Dict[str, Any]) -> Dict[str, Any]:
        """
        判断 Agent 提出的修复动作是否篡改了用户的原始意图目标 (Goal Mutation)
        """
        # 如果改变了关键业务动作的主体数值参数，触发 Guard 拦截
        if original_goal.get("action") == proposed_action.get("action"):
            # 校验核心业务参数变动 (如: 请假天数、转账金额)
            for key in ["days", "amount", "target_recipient"]:
                if key in original_goal and key in proposed_action:
                    if original_goal[key] != proposed_action[key]:
                        logger.warning(f"🛑 [Goal Mutation Guard] Intercepted! Parameter '{key}' mutated from {original_goal[key]} to {proposed_action[key]}")
                        return {
                            "decision": PolicyDecision.REQUIRE_USER_CONFIRMATION,
                            "reason": f"Agent 试图自动将用户目标中的 [{key}: {original_goal[key]}] 篡改为 [{key}: {proposed_action[key]}]。需向用户确认。",
                            "proposed_action": proposed_action
                        }
        
        # 仅仅是补充查询或修改执行路径 (Path Repair)
        logger.info("✅ [Goal Mutation Guard] Policy Check Passed: Proposed action is path repair only.")
        return {"decision": PolicyDecision.ALLOW}

# ============================================================================
# 4. 测试与架构验证 (Test Cases)
# ============================================================================

def run_day35_tests():
    print("=" * 70)
    print("🧪 Scenario 1: Employee Search -> Business Outcome: EMPTY (ToolResult.ok == True)")
    print("=" * 70)
    res_empty = search_employee("Alice")
    print(f"ToolResult.ok        : {res_empty.ok}")
    print(f"BusinessStatus       : {res_empty.data.status}")
    print(f"Data Payload         : {res_empty.data.data}\n")

    print("=" * 70)
    print("🧪 Scenario 2: Employee Search -> Business Outcome: AMBIGUOUS (ToolResult.ok == True)")
    print("=" * 70)
    res_ambiguous = search_employee("Kevin")
    print(f"ToolResult.ok        : {res_ambiguous.ok}")
    print(f"BusinessStatus       : {res_ambiguous.data.status}")
    print(f"Candidates Count     : {len(res_ambiguous.data.data)}")
    print(f"Reason               : {res_ambiguous.data.reason}\n")

    print("=" * 70)
    print("🧪 Scenario 3: Payment Declined -> Business Outcome: DECLINED (ToolResult.ok == True)")
    print("=" * 70)
    res_declined = process_payment(account_id="ACC_8088", amount=5000.0)
    print(f"ToolResult.ok        : {res_declined.ok}")
    print(f"BusinessStatus       : {res_declined.data.status}")
    print(f"Decline Reason       : {res_declined.data.reason}")
    print(f"Is Runtime Failure?  : {not res_declined.ok} (No runtime retry should be triggered!)\n")

    print("=" * 70)
    print("🧪 Scenario 4: Payment Timeout -> Tool Runtime Failure (ToolResult.ok == False)")
    print("=" * 70)
    res_timeout = process_payment(account_id="ACC_8088", amount=100.0, simulate_timeout=True)
    print(f"ToolResult.ok        : {res_timeout.ok}")
    print(f"Error Kind           : {res_timeout.error.kind}")
    print(f"Retryable Flag       : {res_timeout.error.retryable}\n")

    print("=" * 70)
    print("🧪 Scenario 5: Goal Mutation Guard Interception")
    print("=" * 70)
    
    user_original_goal = {"action": "submit_leave", "days": 20, "emp_id": "EMP_1001"}
    
    # 案例 A: Agent 遭到拒绝后，擅自改为申请 5 天 (Goal Mutation)
    agent_mutated_proposal = {"action": "submit_leave", "days": 5, "emp_id": "EMP_1001"}
    policy_res_a = GoalMutationGuard.evaluate(user_original_goal, agent_mutated_proposal)
    print(f"Case A (20天 -> 5天) Policy Decision : {policy_res_a['decision']}")
    print(f"Details                             : {policy_res_a.get('reason')}\n")
    
    # 案例 B: Agent 遭到拒绝后，决定先查询余额或向用户解释 (Path Repair)
    agent_path_repair_proposal = {"action": "query_leave_balance", "emp_id": "EMP_1001"}
    policy_res_b = GoalMutationGuard.evaluate(user_original_goal, agent_path_repair_proposal)
    print(f"Case B (改为前置查询) Policy Decision : {policy_res_b['decision']}")

if __name__ == "__main__":
    run_day35_tests()

2026-09-21 16:35:39,948 - [INFO] - ⚙️ [Tool Execution] search_employee(name='Alice')
2026-09-21 16:35:39,949 - [INFO] - ⚙️ [Tool Execution] search_employee(name='Kevin')
2026-09-21 16:35:39,950 - [INFO] - ⚙️ [Tool Execution] process_payment(account_id='ACC_8088', amount=5000.0)
2026-09-21 16:35:39,952 - [WARNING] - ⚠️ [Business Outcome] Payment Declined due to Insufficient Funds
2026-09-21 16:35:39,953 - [INFO] - ⚙️ [Tool Execution] process_payment(account_id='ACC_8088', amount=100.0)
2026-09-21 16:35:39,953 - [ERROR] - 💥 [Runtime Fault] Payment Gateway Socket Timeout Exception!
2026-09-21 16:35:39,955 - [WARNING] - 🛑 [Goal Mutation Guard] Intercepted! Parameter 'days' mutated from 20 to 5
2026-09-21 16:35:39,957 - [INFO] - ✅ [Goal Mutation Guard] Policy Check Passed: Proposed action is path repair only.


🧪 Scenario 1: Employee Search -> Business Outcome: EMPTY (ToolResult.ok == True)
ToolResult.ok        : True
BusinessStatus       : BusinessStatus.EMPTY
Data Payload         : []

🧪 Scenario 2: Employee Search -> Business Outcome: AMBIGUOUS (ToolResult.ok == True)
ToolResult.ok        : True
BusinessStatus       : BusinessStatus.AMBIGUOUS
Candidates Count     : 2
Reason               : 找到多个同名员工

🧪 Scenario 3: Payment Declined -> Business Outcome: DECLINED (ToolResult.ok == True)
ToolResult.ok        : True
BusinessStatus       : BusinessStatus.DECLINED
Decline Reason       : INSUFFICIENT_FUNDS
Is Runtime Failure?  : False (No runtime retry should be triggered!)

🧪 Scenario 4: Payment Timeout -> Tool Runtime Failure (ToolResult.ok == False)
ToolResult.ok        : False
Error Kind           : ToolErrorKind.TIMEOUT
Retryable Flag       : True

🧪 Scenario 5: Goal Mutation Guard Interception
Case A (20天 -> 5天) Policy Decision : PolicyDecision.REQUIRE_USER_CONFIRMATION
Details               